# Temporal Feature Engineering

When dealing with time, we have to translate dates into actionable numbers. We generally break this down into four strategies:
1. **Deconstruction**: Breaking a single datetime into its component parts (Month, Day, Hour).
2. **Recency (Elapsed Time)**: Calculating the time between two events.
3. **Cyclical Encoding**: Teaching the model that December 31st and January 1st are actually right next to each other.
4. **Lags & Rolling Windows**: Looking backward in time to capture recent momentum (crucial for time-series forecasting).

Let's set up a Python sandbox with some raw e-commerce transaction data!

In [1]:
import pandas as pd
import numpy as np

# Create a dataset of customer purchases
data = {
    'transaction_id': [1, 2, 3, 4, 5],
    'customer_id': [101, 102, 101, 103, 104],
    # Notice these are just strings!
    'purchase_time': ['2023-11-24 08:30:00', '2023-12-31 23:45:00', '2024-01-01 00:15:00', '2024-02-14 18:00:00', '2024-07-04 12:00:00'],
    'signup_date': ['2023-01-15', '2023-12-30', '2023-01-15', '2021-05-20', '2024-07-04']
}

df = pd.DataFrame(data)

# 1. THE MOST IMPORTANT FIRST STEP: Convert strings to actual Pandas Datetime objects!
df['purchase_time'] = pd.to_datetime(df['purchase_time'])
df['signup_date'] = pd.to_datetime(df['signup_date'])

print("✅ Data loaded and converted to Datetime objects!")
display(df)

✅ Data loaded and converted to Datetime objects!


,transaction_id,customer_id,purchase_time,signup_date
0,1,101,2023-11-24 08:30:00,2023-01-15
1,2,102,2023-12-31 23:45:00,2023-12-30
2,3,101,2024-01-01 00:15:00,2023-01-15
3,4,103,2024-02-14 18:00:00,2021-05-20
4,5,104,2024-07-04 12:00:00,2024-07-04


# 1. Deconstructing the Timestamp
A single timestamp contains a massive amount of hidden information. We can use Pandas' `.dt` accessor to extract these pieces into separate columns.

In [2]:
# Create a copy
df_time = df.copy()

# Extract basic components
df_time['purchase_month'] = df_time['purchase_time'].dt.month
df_time['purchase_day_of_week'] = df_time['purchase_time'].dt.dayofweek # Monday=0, Sunday=6
df_time['purchase_hour'] = df_time['purchase_time'].dt.hour

# Create a Boolean Flag using the Day of Week (Is it the weekend?)
# If dayofweek is 5 (Sat) or 6 (Sun), it returns True (1)
df_time['is_weekend'] = (df_time['purchase_time'].dt.dayofweek >= 5).astype(int)

print("--- Data after Deconstruction ---")
display(df_time[['purchase_time', 'purchase_month', 'purchase_hour', 'purchase_day_of_week', 'is_weekend']])

--- Data after Deconstruction ---


,purchase_time,purchase_month,purchase_hour,purchase_day_of_week,is_weekend
0,2023-11-24 08:30:00,11,8,4,0
1,2023-12-31 23:45:00,12,23,6,1
2,2024-01-01 00:15:00,1,0,0,0
3,2024-02-14 18:00:00,2,18,2,0
4,2024-07-04 12:00:00,7,12,3,0


*(By breaking it apart, the model can now independently discover that sales spike on Weekends (`is_weekend = 1`), regardless of what month it is!)*

# 2. Calculating Recency (Elapsed Time)
Algorithms don't inherently understand the concept of "Tenure" or "Recency" from raw dates. We have to do the subtraction for them. 

Often, we want to know how long a customer has been with us before they made a specific purchase.

In [3]:
# Calculate the time difference between Purchase and Signup
df_time['days_since_signup'] = (df_time['purchase_time'] - df_time['signup_date']).dt.days

print("--- Data with Recency Feature ---")
display(df_time[['customer_id', 'signup_date', 'purchase_time', 'days_since_signup']])

--- Data with Recency Feature ---


,customer_id,signup_date,purchase_time,days_since_signup
0,101,2023-01-15,2023-11-24 08:30:00,313
1,102,2023-12-30,2023-12-31 23:45:00,1
2,101,2023-01-15,2024-01-01 00:15:00,351
3,103,2021-05-20,2024-02-14 18:00:00,1000
4,104,2024-07-04,2024-07-04 12:00:00,0


*(Now the model has a powerful new feature: `days_since_signup`. It might discover that customers who have been around for > 300 days buy much more expensive items!)*

# 3. Cyclical Encoding (The Expert Trick)
Look closely at Customer 102 and Customer 101. 
* Customer 102 bought an item on **December 31st** (Month 12).
* Customer 101 bought an item 30 minutes later on **January 1st** (Month 1).

To a human, these purchases happened at the exact same time of year. But to a Machine Learning model, the difference between Month 12 and Month 1 is mathematically massive. The model thinks they are 11 months apart!

To fix this, we map our time data onto a circle using **Sine and Cosine** transformations.

In [4]:
# We have 12 months in a year. We want to wrap them around a circle.
months_in_year = 12

# Calculate Sine and Cosine for the month
df_time['month_sin'] = np.sin(2 * np.pi * df_time['purchase_month'] / months_in_year)
df_time['month_cos'] = np.cos(2 * np.pi * df_time['purchase_month'] / months_in_year)

print("--- Cyclical Encoding for Months ---")
display(df_time[['purchase_time', 'purchase_month', 'month_sin', 'month_cos']])

# Notice how December (Month 12) and January (Month 1) now share very similar coordinates 
# on the Sin/Cos circle, proving to the model that they are neighbors!

--- Cyclical Encoding for Months ---


,purchase_time,purchase_month,month_sin,month_cos
0,2023-11-24 08:30:00,11,-5.000000e-01,0.866025
1,2023-12-31 23:45:00,12,-2.449294e-16,1.000000
2,2024-01-01 00:15:00,1,5.000000e-01,0.866025
3,2024-02-14 18:00:00,2,8.660254e-01,0.500000
4,2024-07-04 12:00:00,7,-5.000000e-01,-0.866025


# 4. Lag Features & Rolling Windows
If you are predicting Daily Sales, what happened *today* is highly dependent on what happened *yesterday*. 

We engineer these time-series signals using **Lags** (shifting data backwards) and **Rolling Windows** (calculating averages over the last N days).

In [5]:
# Let's create a quick dataset of daily sales for a single store
sales_data = pd.DataFrame({
    'date': pd.date_range(start='2024-01-01', periods=5, freq='D'),
    'daily_sales': [100, 150, 120, 200, 180]
})

# 1. LAG FEATURE: What were the sales exactly 1 day ago?
# .shift(1) pushes all the data down by one row
sales_data['sales_yesterday'] = sales_data['daily_sales'].shift(1)

# 2. ROLLING WINDOW: What is the average sales over the last 3 days?
# min_periods=1 ensures it still calculates an average even on day 1 or 2
sales_data['3_day_rolling_avg'] = sales_data['daily_sales'].rolling(window=3, min_periods=1).mean()

print("--- Lags and Rolling Windows ---")
display(sales_data)

--- Lags and Rolling Windows ---


,date,daily_sales,sales_yesterday,3_day_rolling_avg
0,2024-01-01,100,NaN,100.000000
1,2024-01-02,150,100.0,125.000000
2,2024-01-03,120,150.0,123.333333
3,2024-01-04,200,120.0,156.666667
4,2024-01-05,180,200.0,166.666667


*(Notice the `NaN` on the first row of `sales_yesterday`? We can't know what the sales were yesterday because our dataset starts today! You will have to use your Missing Value Treatment skills from Lesson 02 to handle those `NaNs` before training a model!)*

---

## Real-World Use Case or Analogy:
Think of Temporal Feature Engineering like **Managing a City Subway System**:

* **Raw Time (The System Log)**: A passenger taps their ticket at the turnstile, and the database records: "08:00 AM, November 1st." By itself, this is just a static statement of fact.
* **Deconstruction**: The traffic analyst doesn't care about the exact timestamp. They care about the extracted features: *"Is it a weekday?"* *"Is it morning rush hour?"* *"Is it a public holiday?"* Those extracted components dictate how many trains they need to deploy to handle the crowd.
* **Recency**: The station manager monitors the platform. They don't look at the exact timestamp the last train left; they calculate the elapsed time: *"It has been exactly 4 minutes since the last train departed, so the platform is getting crowded."*
* **Cyclical Nature**: The night-shift conductor knows that 11:59 PM (December 31st) and 12:01 AM (January 1st) are only two minutes apart. The physical train keeps moving smoothly along the tracks, maintaining its momentum and completely ignoring the fact that the calendar's numbers just violently reset to a brand new day, month, and year. 

---